In [2]:

# ==============================================================================
# STEP 0: ONE-BY-ONE DEPENDENCY INSTALLATION & ENVIRONMENT SETUP
# ==============================================================================
# -------------------------
# 1. Install libraries
# -------------------------
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets
!pip -q install accelerate
!pip -q install peft
!pip -q install bitsandbytes
!pip -q install torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 762.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [1]:
# ==============================================================================
# STAGE 1: NON-INSTRUCTION FINE-TUNING (Domain Adaptation Workflow)
# ==============================================================================

# ------------------------------------------------------------------------------
# 0. REQUIRED IMPORTS
# ------------------------------------------------------------------------------
import os
import re
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from transformers import Trainer, DataCollatorForLanguageModeling

# ------------------------------------------------------------------------------
# 1 & 2. LOADING, CLEANING, AND CHUNKING RAW DOMAIN TEXT
# ------------------------------------------------------------------------------
def clean_and_chunk_text(file_path, chunk_size=300, overlap=30):
    """
    Reads a raw text file from the repository layout, applies standard cleaning,
    and splits the text into fixed word counts with sliding window overlaps to
    preserve continuous domain knowledge context.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Could not find domain data at '{file_path}'. "
            f"Please verify that your workspace setup or option script has executed correctly."
        )

    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Text Cleaning Pipeline
    cleaned_text = re.sub(r'\s+', ' ', raw_text)            # Normalize arbitrary spacing/newlines
    cleaned_text = re.sub(r'[^\x00-\x7F]+', '', cleaned_text) # Exclude non-ASCII code character defects
    words = cleaned_text.strip().split()

    # Overlap Window Chunking Logic
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i : i + chunk_size]
        chunks.append(" ".join(chunk_words))
        i += (chunk_size - overlap)
        if i + overlap >= len(words): # Break early if remaining content is too minimal
            break

    # Wrap text array chunks into a structured Hugging Face Dataset format
    dataset = Dataset.from_dict({"text": chunks})
    print(f"✅ Preprocessing Complete: Generated {len(dataset)} structural chunks.")
    return dataset

# Load your prepared data from the assignment folder structure
print("Step 1 & 2: Loading, cleaning, and chunking text data...")
train_dataset = clean_and_chunk_text("/content/drive/MyDrive/domain-ai-assistant-finetuning/data/non_instruction_data.txt", chunk_size=300, overlap=30)

# ------------------------------------------------------------------------------
# 3. LOADING BASE MODEL USING UNSLOTH
# ------------------------------------------------------------------------------
print("\nStep 3: Initializing base model with Unsloth...")
max_seq_length = 2048
dtype = None          # Automatically defaults to correct hardware config (e.g. float16/bfloat16)
load_in_4bit = True   # Enforces 4-bit quantization layout to maximize VRAM room on free T4s

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",  # Modern, lightweight base model footprint
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# ------------------------------------------------------------------------------
# 4. APPLYING LORA / QLORA PARAMS
# ------------------------------------------------------------------------------
print("\nStep 4: Wrapping the base model with PEFT/LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # Bottleneck rank sizing for domain parameter shifts
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,      # Balanced standard alpha scale
    lora_dropout = 0,     # 0 is mathematically optimized for Unsloth fast kernels
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial parameter setting to reduce activation memory overhead
    random_state = 3407,
)
# ------------------------------------------------------------------------------
# 5. TRAINING ON RAW TEXT (Bypassing SFTConfig Pickling Bug)
# ------------------------------------------------------------------------------
print("\nStep 5: Starting fine-tuning using an optimized Trainer routine...")

# 1. Use a standard language modeling data collator to handle the raw text
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 2. Tokenize the text dataset mapping completely ahead of time
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=max_seq_length)

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 3. Use the stable base Trainer class directly to bypass trl's SFTConfig pickling conflict
trainer = Trainer(
    model = model,
    train_dataset = tokenized_dataset,
    data_collator = data_collator,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "/content/drive/MyDrive/domain-ai-assistant-finetuning/outputs/stage1_raw_ft",
        report_to = "none"
    ),
)

print("🚀 Commencing execution loop...")
trainer_stats = trainer.train()

# ------------------------------------------------------------------------------
# 6. SAVING THE ADAPTER / MODEL
# ------------------------------------------------------------------------------
print("\nStep 6: Saving Stage 1 adapter configuration...")
output_adapter_path = "/content/drive/MyDrive/domain-ai-assistant-finetuning/models/stage1_adapter"
model.save_pretrained(output_adapter_path)
tokenizer.save_pretrained(output_adapter_path)
print(f"💾 Weights securely logged at: '{output_adapter_path}'")

# ------------------------------------------------------------------------------
# 7. TESTING THE MODEL AFTER NON-INSTRUCTION FINE-TUNING
# ------------------------------------------------------------------------------
print("\n" + "="*60 + "\nStep 7: Executing Text Completion Verification Inference Pipeline...\n" + "="*60)

# Shift active configuration to hardware inference speedups
FastLanguageModel.for_inference(model)

# Evaluation prefix using an excerpt from the provided customer service parameters
test_prefix = "Our standard customer support protocol dictates that all refund requests must be"

inputs = tokenizer([test_prefix], return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    use_cache=True,
    temperature=0.7,
    top_p=0.9
)

decoded_result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print(f"Inference Prompt Phrase:\n-> {test_prefix}\n")
print(f"Model Continuous Domain Completion Output:\n-> {decoded_result}\n")
print("="*60 + "\nStage 1 notebook compilation sequence executed successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Step 1 & 2: Loading, cleaning, and chunking text data...
✅ Preprocessing Complete: Generated 7 structural chunks.

Step 3: Initializing base model with Unsloth...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Step 4: Wrapping the base model with PEFT/LoRA adapters...


Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



Step 5: Starting fine-tuning using an optimized Trainer routine...


Map:   0%|          | 0/7 [00:00<?, ? examples/s]

🚀 Commencing execution loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.645900
2,3.645900
3,3.628100
4,3.556300
5,3.444000
6,3.291200
7,3.092900
8,2.879000
9,2.647500
10,2.401600



Step 6: Saving Stage 1 adapter configuration...
💾 Weights securely logged at: '/content/drive/MyDrive/domain-ai-assistant-finetuning/models/stage1_adapter'

Step 7: Executing Text Completion Verification Inference Pipeline...
Inference Prompt Phrase:
-> Our standard customer support protocol dictates that all refund requests must be

Model Continuous Domain Completion Output:
-> Our standard customer support protocol dictates that all refund requests must be evaluated within three business days of submission. Once a return package is checked into our warehouse repository, an automated webhook notifies the financial ledger system to reverse the original authorization line item. For customer safety and logistical efficiency, return shipments require a unique Return Merchandise Authorization tracking number generated exclusively through our portal. Parcels arriving at our logistics center without a verified barcoded invoice are flagged for manual exception routing. When a duplicate payme